# 🚀 GE2PE Persian Diacritization - Google Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elikaaghaei/Rahnema_college_phonemizer_v1/blob/main/GE2PE_Colab_Demo.ipynb)

این Notebook پروژه GE2PE را در Google Colab اجرا می‌کند.

## مزایا:
- ✅ GPU رایگان
- ✅ RAM زیاد (12GB+)
- ✅ فضای نامحدود
- ✅ بدون نیاز به نصب local

## مراحل:
1. Setup محیط
2. Clone repository
3. نصب dependencies
4. تست Dataset Loader
5. تست Tokenizer
6. اجرای API (با ngrok)
7. تست API

## 1️⃣ بررسی محیط و GPU

In [ ]:
# بررسی GPU
!nvidia-smi

# بررسی فضای دیسک
!df -h | grep -E "Filesystem|/content"

# بررسی RAM
!free -h

## 2️⃣ Clone Repository

In [ ]:
# Clone repository
!git clone https://github.com/elikaaghaei/Rahnema_college_phonemizer_v1.git
%cd Rahnema_college_phonemizer_v1

# نمایش ساختار پروژه
!ls -la

## 3️⃣ نصب Dependencies

In [ ]:
# نصب PyTorch (Colab از قبل نصب دارد، اما برای اطمینان)
!pip install -q torch torchvision torchaudio

# نصب API dependencies
!pip install -q -r api/requirements.txt

# نصب data dependencies
!pip install -q -r data/requirements.txt

# نصب Parsivar برای GE2PE
!pip install -q parsivar

# نصب pyngrok برای expose کردن API
!pip install -q pyngrok

print("✅ همه dependencies نصب شدند!")

In [ ]:
# بررسی نصب PyTorch و GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4️⃣ تست Dataset Loader

In [ ]:
from data.loader import PhonemizerDataset, train_val_split, collate_fn
from torch.utils.data import DataLoader

# بارگذاری dataset (نمونه کوچک برای تست)
print("📖 بارگذاری dataset...")
dataset = PhonemizerDataset(
    csv_path='phonemizer _dataset_v1.csv/phonemizer _dataset_v1.csv',
    mode='char',
    preserve_diacritics=True,
    max_samples=100  # فقط 100 نمونه برای تست
)

print(f"\n✅ Dataset loaded: {len(dataset)} samples")

# نمایش نمونه‌ها
print("\n📋 نمونه‌های dataset:")
for i in range(3):
    sample = dataset[i]
    print(f"\nنمونه {i+1}:")
    print(f"  ورودی: {sample['text'][:50]}")
    print(f"  خروجی: {sample['phonemes'][:50]}")
    print(f"  تعداد توکن: {len(sample['tokens'])}")

In [ ]:
# تست train/val split
train_dataset, val_dataset = train_val_split(dataset, val_frac=0.2, seed=42)

print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")

# ساخت DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# نمایش یک batch
batch = next(iter(train_loader))
print(f"\n📦 Batch info:")
print(f"  Batch size: {batch['batch_size']}")
print(f"  Lengths: {batch['lengths']}")
print(f"  Sample text: {batch['texts'][0][:50]}...")

## 5️⃣ تست Tokenizer

In [ ]:
from data.tokenizer import PersianTokenizer

# ایجاد tokenizer
tokenizer = PersianTokenizer()

# Build vocab
texts = ['سلام دنیا', 'این یک تست است', 'زبان فارسی زیباست']
tokenizer.build_vocab(texts, mode='char', min_freq=1)

print(f"\n✅ Vocab size: {tokenizer.vocab_size}")
print(f"\nSpecial tokens:")
for token, id in list(tokenizer.vocab.items())[:10]:
    print(f"  '{token}' → {id}")

In [ ]:
# تست encode/decode
text = "سلام"
print(f"Original: {text}")

# Tokenize
tokens = tokenizer.tokenize(text, mode='char')
print(f"Tokens: {tokens}")

# Encode
encoded = tokenizer.encode(text, mode='char')
print(f"Encoded: {encoded}")

# Decode
decoded = tokenizer.decode(encoded)
print(f"Decoded: {decoded}")
print(f"Match: {text == decoded} ✅" if text == decoded else "Match: ❌")

## 6️⃣ تست GE2PE Model (اختیاری)

In [ ]:
# نکته: برای استفاده از GE2PE واقعی، نیاز به model checkpoint دارید
# اگر checkpoint ندارید، از model base استفاده می‌شود

try:
    from GE2PE.GE2PE import GE2PE
    
    print("🔄 بارگذاری GE2PE model...")
    print("⚠️  از model base استفاده می‌شود (checkpoint موجود نیست)")
    
    # استفاده از model base (کوچک)
    model = GE2PE(
        model_path='google/mt5-small',
        GPU=torch.cuda.is_available()
    )
    
    # تست
    test_text = "سلام دنیا"
    result = model.generate([test_text], batch_size=1)
    
    print(f"\n✅ Model test:")
    print(f"  Input:  {test_text}")
    print(f"  Output: {result[0]}")
    
except Exception as e:
    print(f"⚠️  Model loading skipped: {e}")
    print("برای استفاده از model، checkpoint را آپلود کنید.")

## 7️⃣ اجرای FastAPI با ngrok

In [ ]:
# نصب ngrok برای expose کردن API
from pyngrok import ngrok
import threading
import uvicorn
import time

# توقف ngrok tunnel قبلی (اگر وجود داشت)
ngrok.kill()

print("🚀 راه‌اندازی FastAPI server...")

In [ ]:
# اجرای API در background thread
def run_api():
    uvicorn.run(
        "api.main:app",
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

# شروع server
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

# صبر برای startup
print("⏳ صبر برای startup...")
time.sleep(10)

print("✅ Server started!")

In [ ]:
# ایجاد ngrok tunnel
public_url = ngrok.connect(8000)

print("="*60)
print("🌐 API در دسترس است!")
print("="*60)
print(f"\n📡 Public URL: {public_url}")
print(f"\n🔗 Endpoints:")
print(f"  - Health: {public_url}/health")
print(f"  - Metrics: {public_url}/metrics")
print(f"  - Docs: {public_url}/docs")
print(f"  - Diacritize: {public_url}/diacritize (POST)")
print("\n" + "="*60)
print("💡 Swagger UI را در browser باز کنید!")
print("="*60)

## 8️⃣ تست API

In [ ]:
import requests
import json

# گرفتن URL
api_url = str(public_url)

print("🧪 تست API endpoints...\n")

In [ ]:
# تست Health endpoint
response = requests.get(f"{api_url}/health")
print("1️⃣ Health Check:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))
print()

In [ ]:
# تست Metrics endpoint
response = requests.get(f"{api_url}/metrics")
print("2️⃣ Metrics:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))
print()

In [ ]:
# تست Diacritization - تک متن
payload = {
    "texts": "سلام دنیا",
    "batch_size": 10,
    "use_rules": False
}

response = requests.post(
    f"{api_url}/diacritize",
    json=payload
)

print("3️⃣ Diacritize (Single):")
if response.status_code == 200:
    result = response.json()
    print(f"Input:  {payload['texts']}")
    print(f"Output: {result['results'][0]}")
    print(f"Time:   {result['processing_time_ms']} ms")
else:
    print(f"Error: {response.status_code}")
    print(response.text)
print()

In [ ]:
# تست Diacritization - چند متن
payload = {
    "texts": [
        "سلام دنیا",
        "این یک تست است",
        "زبان فارسی زیباست"
    ],
    "batch_size": 10
}

response = requests.post(
    f"{api_url}/diacritize",
    json=payload
)

print("4️⃣ Diacritize (Batch):")
if response.status_code == 200:
    result = response.json()
    print(f"Processed: {result['count']} texts")
    print(f"Time: {result['processing_time_ms']} ms\n")
    
    for i, (inp, out) in enumerate(zip(payload['texts'], result['results']), 1):
        print(f"{i}. Input:  {inp}")
        print(f"   Output: {out}")
        print()
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## 9️⃣ Interactive Testing

In [ ]:
# تست تعاملی - متن خودتان را وارد کنید
from IPython.display import display, HTML

def diacritize_text(text):
    """تابع helper برای diacritization"""
    response = requests.post(
        f"{api_url}/diacritize",
        json={"texts": text}
    )
    
    if response.status_code == 200:
        result = response.json()
        return result['results'][0]
    else:
        return f"Error: {response.status_code}"

# مثال‌ها
examples = [
    "من به مدرسه می روم",
    "کتاب خوب است",
    "هوا امروز خوب است"
]

print("📝 مثال‌های diacritization:\n")
for text in examples:
    result = diacritize_text(text)
    print(f"ورودی:  {text}")
    print(f"خروجی: {result}")
    print()

In [ ]:
# فرم تعاملی برای تست
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output

# ایجاد widgets
text_input = widgets.Textarea(
    value='',
    placeholder='متن فارسی خود را اینجا وارد کنید...',
    description='ورودی:',
    layout=Layout(width='80%', height='100px')
)

button = widgets.Button(
    description='اعراب‌گذاری کن',
    button_style='success',
    icon='check'
)

output = widgets.Output()

def on_button_click(b):
    with output:
        clear_output()
        text = text_input.value
        if text:
            print(f"🔄 در حال پردازش...")
            result = diacritize_text(text)
            print(f"\n✅ نتیجه:")
            print(f"   {result}")
        else:
            print("⚠️  لطفاً متنی وارد کنید")

button.on_click(on_button_click)

display(text_input, button, output)

## 🔟 خلاصه و لینک‌های مفید

In [ ]:
print("="*70)
print("📊 خلاصه Deployment")
print("="*70)
print(f"\n🌐 API Public URL: {public_url}")
print(f"\n📖 مستندات:")
print(f"   - Swagger UI: {public_url}/docs")
print(f"   - ReDoc: {public_url}/redoc")
print(f"\n🔗 Endpoints:")
print(f"   GET  {public_url}/health")
print(f"   GET  {public_url}/metrics")
print(f"   POST {public_url}/diacritize")
print(f"\n💡 نکات:")
print(f"   - API در background در حال اجرا است")
print(f"   - برای توقف: Runtime → Interrupt execution")
print(f"   - ngrok tunnel تا 2 ساعت فعال می‌ماند")
print(f"\n🎓 برای استفاده در Python:")
print(f"   import requests")
print(f"   response = requests.post('{public_url}/diacritize',")
print(f"                            json={{'texts': 'سلام'}})")
print(f"   print(response.json())")
print("\n" + "="*70)
print("✅ همه چیز آماده است! API را از browser تست کنید.")
print("="*70)

## 📚 منابع و Next Steps

### Repository:
- GitHub: https://github.com/elikaaghaei/Rahnema_college_phonemizer_v1

### مستندات:
- API README: `api/README.md`
- Docker Guide: `DOCKER_GUIDE.md`

### Next Steps:
1. Training model با dataset کامل
2. Fine-tuning MT5 برای Persian diacritization
3. Deployment روی Cloud (AWS/GCP/Azure)
4. ایجاد Web UI با Gradio/Streamlit
5. Integration با Telegram Bot

### مشارکت:
- Issues: https://github.com/elikaaghaei/Rahnema_college_phonemizer_v1/issues
- Pull Requests: خوشحال می‌شویم!

---

**🎉 موفق باشید!**